# CreditLens — 04 · Evaluation & Calibration

Evaluate the **saved, calibrated** models (`models/*.joblib`, produced by `creditlens/pipeline.py` /
`make train`). The frontend lets the user pick any of the 6, so all are persisted and scored here.

We look at: the leaderboard (AUC/KS/Gini), **calibration** (reliability curves + ECE before/after
isotonic — credit needs trustworthy PD), **ROC**, and **lift by decile**.

## 0 · Load saved models + metadata, rebuild the test split

In [1]:
import sys; sys.path.insert(0, '..')
import json
import joblib
import numpy as np
import pandas as pd
from warnings import filterwarnings; filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve

from creditlens.config import MODELS_DIR, TARGET, RANDOM_SEED
from creditlens.data.features import load_or_build_model_matrix, MODEL_FEATURES
from creditlens.evaluation.metrics import lift_table, summarize

meta = json.loads((MODELS_DIR / 'metadata.json').read_text())
models = {p.stem: joblib.load(p) for p in sorted(MODELS_DIR.glob('*.joblib'))}
print('loaded models:', list(models))
print('best by AUC:', meta['best'])

# Rebuild the SAME held-out test split the pipeline used (same seed + sizes => identical rows).
mat = load_or_build_model_matrix()
X, y = mat[MODEL_FEATURES], mat[TARGET]
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED)
_, X_te, _, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_SEED)
print('test set:', X_te.shape)

loaded models: ['catboost', 'lgbm', 'logreg', 'rf', 'stacking', 'xgb']
best by AUC: stacking
test set: (46127, 15)


## 1 · Leaderboard (calibrated, on held-out test)
AUC/KS/Gini per model, plus ECE before vs after calibration (lower = better-calibrated PD).

In [2]:
rows = []
proba = {}
for name, m in models.items():
    p = m.predict_proba(X_te)[:, 1]
    proba[name] = p
    s = summarize(y_te, p)
    rows.append({'model': name, **{k: round(v, 4) for k, v in s.items()},
                 'ece_uncal': meta['models'][name]['ece_uncalibrated']})
board = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
board

,model,auc,gini,ks,ece,ece_uncal
0,stacking,0.7432,0.4865,0.3677,0.0022,0.0041
1,lgbm,0.7429,0.4857,0.3682,0.0017,0.3309
2,xgb,0.7428,0.4856,0.3660,0.0019,0.3418
3,catboost,0.7414,0.4829,0.3645,0.0010,0.3428
4,rf,0.7350,0.4701,0.3545,0.0026,0.2767
5,logreg,0.7276,0.4553,0.3401,0.0016,0.3534


## 2 · Reliability (calibration) curves
Bin predictions, plot mean predicted PD vs observed default rate. On the diagonal = perfectly calibrated.

In [3]:
import plotly.graph_objects as go

def reliability(y_true, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    xs, ys = [], []
    for b in range(n_bins):
        m = idx == b
        if m.any():
            xs.append(p[m].mean()); ys.append(np.asarray(y_true)[m].mean())
    return xs, ys

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 0.6], y=[0, 0.6], mode='lines', name='perfect',
                         line=dict(dash='dash', color='grey')))
for name in board['model']:
    xs, ys = reliability(y_te, proba[name])
    fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines+markers', name=name))
fig.update_layout(template='plotly_white', width=620, height=520,
                  title='Reliability curves (calibrated)', xaxis_title='mean predicted PD',
                  yaxis_title='observed default rate')
fig.show()

## 3 · ROC curves

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='chance',
                         line=dict(dash='dash', color='grey')))
for name in board['model']:
    fpr, tpr, _ = roc_curve(y_te, proba[name])
    auc = board.loc[board['model'] == name, 'auc'].iloc[0]
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} ({auc:.3f})'))
fig.update_layout(template='plotly_white', width=620, height=520, title='ROC curves',
                  xaxis_title='false positive rate', yaxis_title='true positive rate')
fig.show()

## 4 · Lift by decile (best model)
Sort by score, split into deciles; lift = decile default rate / base rate. The top decile catching
several× the base rate is what makes the score useful for prioritizing review.

In [5]:
best = meta['best']
lt = lift_table(y_te, proba[best]).reset_index()
fig = go.Figure(go.Bar(x=lt['decile'], y=lt['lift'], marker_color='#2563eb'))
fig.add_hline(y=1.0, line_dash='dash', line_color='grey')
fig.update_layout(template='plotly_white', width=620, height=420,
                  title=f'Lift by score decile — {best} (0 = riskiest)',
                  xaxis_title='score decile', yaxis_title='lift vs base rate')
fig.show()
lt.round(4)

,decile,n,defaults,rate,lift
0,0,4613,1241,0.2690,3.3322
1,1,4613,656,0.1422,1.7614
2,2,4613,461,0.0999,1.2378
3,3,4612,339,0.0735,0.9104
4,4,4613,273,0.0592,0.7330
5,5,4613,238,0.0516,0.6391
6,6,4612,178,0.0386,0.4781
7,7,4613,160,0.0347,0.4296
8,8,4613,108,0.0234,0.2900
9,9,4612,70,0.0152,0.1880


# Notebook summary & key insights

## Task
Evaluate and calibrate the 6 saved models on a held-out test set; confirm trustworthy PD and finalize the
model-card numbers. All 6 are persisted (`models/*.joblib`) because the frontend lets the user choose.

## Setup
- Models trained + isotonic-calibrated by `creditlens/pipeline.py` (70/15/15 train/calib/test).
- Metrics: ROC AUC, KS, Gini, ECE (before/after calibration), reliability curves, lift by decile.

## Findings
- _From the leaderboard:_ boosters lead (~0.74 AUC); read exact AUC/KS/ECE from the table.
- **Calibration:** isotonic pulls ECE down (boosters with `scale_pos_weight` are over-confident raw) —
  reliability curves sit on the diagonal after calibration. This is what makes the served PD trustworthy.
- **Lift:** the riskiest decile catches several× the base default rate — the score is decision-useful.

## Insights & Recommendations
- **Insight:** AUC alone is not enough for credit — an over-confident 0.74-AUC model gives bad PDs;
  calibration is mandatory before serving.
- **Insight:** Rebuilding the split from the same seed reproduces the exact test set without storing it.
- **Recommendation:** Serve **calibrated** models; expose per-model AUC from `metadata.json` in the UI selector.
- **Recommendation:** Pick the operating threshold from lift / cost of false-approve vs false-reject, not 0.5.

## Next
Phase **5 · Serving** — FastAPI `/predict` loads `models/<name>.joblib` (user-selected), returns calibrated
PD + band + decision over the 15-feature contract; wire `app/CreditLens.html` to it.